# Intro to CometMirror
This notebook is all about running and configuring CometMirror. In this notebook, you'll learn how to:
1) Load up a pre-defined scenario and run a simulation.
2) Specify custom time-dependent trajectories for simulations (e.g. current ramps, impurity injections)
3) Save simulation results and Xarray or Pandas dataframes.
4) Provided visualization tools
5) Generate random walks for a subset of the parameters
6) Configure batches of simulation runs using `MultiCases` and `CombinatorialCases`

Let's begin by loading CometMirror for the SPARC PRD and print out the initial `State` and `Params`.

In [149]:
%load_ext autoreload
%autoreload 2

import warnings
from pprint import pprint

import popsim.config as config
from popsim.scenarios.sparc_prd.comet_mirror import build_comet_mirror_config

# Ignore the xarray warning about stripping away units.
warnings.filterwarnings("ignore", message="The unit of the quantity is stripped when downcasting to ndarray")

# Initialize the simulator.
model, state, params = build_comet_mirror_config()

# Make a time base for all of our simulations.
time_base = config.make_time_base(t0=0.0, t1=5.0, dt=0.01)

pprint(state)
pprint(params)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
State(stored_energy=23.0,
      density_state=State(vol_avg_ion={<FuelSpecies.Tritium: -3>: 13.5,
                                       <FuelSpecies.Deuterium: -2>: 13.5,
                                       <Impurity.Helium: 2>: 1.6199999999999999,
                                       <Impurity.Oxygen: 8>: 0.0837,
                                       <Impurity.Argon: 18>: 0.0,
                                       <Impurity.Tungsten: 74>: 0.00040500000000000003}),
      hmode_state=State(hmode=f64[]))
Params(magnetic_field_on_axis=array(12.2),
       plasma_current=array(8700000.),
       fraction_of_external_power_coupled=array(0.9),
       normalized_inverse_temp_scale_length=array(2.5),
       electron_density_peaking_offset=array(-0.1),
       ion_density_peaking_offset=array(-0.2),
       temperature_peaking=array(2.5),
       ion_to_electron_temp_ratio=array(1.),
       confinement_ti

## Simulating with State + Params

The `State` dataclass is a vector of variables that are being simulated, while `Params` can be thought of as boundary conditions and/or assumptions. Letting $\mathbf{x}_t$ denote the state at time $t$ and $\mathbf{p}_t$ denote the params at time $t$, at every time step of the simulation, what essentially happens is something like this:
$$\mathbf{x}_{t+1} = f(\mathbf{x}_t, \mathbf{p}_t)$$

Below, you will see that we call the `simulate` function like this:
```python
dataset = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=new_params,
)
```
so, in this API, the `model` is sorta like the $f$ function, the `time_base` is the set of time steps we wish to simulate for, the `initial_state` is the initial state $\mathbf{x}_0$, and the `params` are $\mathbf{p}_t$, where some elements can be static and others can be time-dependent.

## The Magical Params Dataclass
The `Params` dataclass is where the magic happens, and where you get super-powers. Every element of it is configurable by you. Let's begin by defining a simulation time-base and defining a current ramp that starts 1 second into the simulation. While we're at it, why not define a auxiliary heating ramp rate, and also a tungsten impurity injection that occurs between (2.0, 2.1) seconds in the simulaiton?

In [150]:
from copy import deepcopy

import jax.numpy as jnp

from popsim.enums import Impurity
from popsim.simulators.comet_mirror.simulate import simulate

# Create a copy of "params". It's best to keep the original one around untouched.
new_params = deepcopy(params)

# Manually define a current-ramp where the key is the time in seconds and the value is the current in Amperes.
new_params.plasma_current = {0.0: 8.7e6, 1.0: 8.7e6, 5.0: 4.0e6}

# Manually define an auxiliary heating power ramp where the key is the time in seconds and the value is the power in MW.
new_params.P_aux_MW = {0.0: 11.1, 5.0: 7.0}

# Manually define a quick tungsten spike. Note that under the hood linear interpolation is happening, so we need this
# perhaps somewhat awkward definition.
new_params.fueling19[Impurity.Tungsten] = {
    0.0: 0.0,
    1.99: 0.0,  # Start ramping impurities.
    2.0: 0.1,  # Impurity injection.
    2.1: 0.1,  # Impurity injection holding.
    2.11: 0.0,  # Impurity drops back to 0.0.
    5.0: 0.0,  # Impurity holds at 0.0.
}

dataset = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=new_params,
)

/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/

## Visualizing Simulation Results
Okay, so we ran a simulation and got a `dataset` object out. By default, this will be an `xarray` dataset, which is a really powerful and useful data analysis package create by geoscientists to help their spatial data problems.

We can view the structure of the data in a Jupyter Notebook by executing the cell below. You'll note that there are **a lot** of variables. This is because `CometMirror` is set up to automatically log all of the variables in the scope of its main body (which if you ask me is pretty cool). Variables that start with `state` are state variables while everything else is under `aux`.

In [151]:
dataset

<xarray.Dataset>
Dimensions:                                                               (
                                                                           time: 501,
                                                                           rho: 30)
Coordinates:
  * time                                                                  (time) float64 ...
Dimensions without coordinates: rho
Data variables: (12/168)
    aux.EV_TO_JOULE                                                       (time) float64 ...
    aux.P_alpha_MW                                                        (time) float64 ...
    aux.P_fusion_MW                                                       (time) float64 ...
    aux.P_neutron_MW                                                      (time) float64 ...
    aux.P_ohmic_MW                                                        (time) float64 ...
    aux.P_rad_MW                                                          (time) float64 ...
    ...                                                                    ...
    state.density_state.vol_avg_ion.Impurity.Helium                       (time) float64 ...
    state.density_state.vol_avg_ion.Impurity.Oxygen                       (time) float64 ...
    state.density_state.vol_avg_ion.Impurity.Argon                        (time) float64 ...
    state.density_state.vol_avg_ion.Impurity.Tungsten                     (time) float64 ...
    state.hmode_state.hmode                                               (time) float64 ...
    state.stored_energy                                                   (time) float64 ...

Now, one can extract the variables you care about and write your own plotting functions, but I've also included a generic plotting function `visualize_time_series` that should make your life a bit easier. Just grab the list of variables you care about plotting like what is shown below, and pass it to the function. Later on, you will learn about an interactive GUI that can also be used.

In [152]:
import holoviews as hv

from popsim.visualize import visualize_time_series

hv.extension("matplotlib")

visualize_vars = [
    "aux.params.plasma_current",
    "aux.params.P_aux_MW",
    "aux.params.fueling19.Impurity.Tungsten",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.hmode_state.hmode",
    "aux.Prad_imp_MW",
    "aux.P_rad_MW",
    "aux.tau_E",
]
visualize_time_series(dataset[visualize_vars])

<img src='data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAYAAACqaXHeAAAABHNCSVQICAgIfAhkiAAAAAlwSFlz
AAAB+wAAAfsBxc2miwAAABl0RVh0U29mdHdhcmUAd3d3Lmlua3NjYXBlLm9yZ5vuPBoAAA6zSURB
VHic7ZtpeFRVmsf/5966taWqUlUJ2UioBBJiIBAwCZtog9IOgjqACsogKtqirT2ttt069nQ/zDzt
tI4+CrJIREFaFgWhBXpUNhHZQoKBkIUASchWla1S+3ar7r1nPkDaCAnZKoQP/D7mnPOe9/xy76n3
nFSAW9ziFoPFNED2LLK5wcyBDObkb8ZkxuaoSYlI6ZcOKq1eWFdedqNzGHQBk9RMEwFAASkk0Xw3
ETacDNi2vtvc7L0ROdw0AjoSotQVkKSvHQz/wRO1lScGModBFbDMaNRN1A4tUBCS3lk7BWhQkgpD
lG4852/+7DWr1R3uHAZVQDsbh6ZPN7CyxUrCzJMRouusj0ipRwD2uKm0Zn5d2dFwzX1TCGhnmdGo
G62Nna+isiUqhkzuKrkQaJlPEv5mFl2fvGg2t/VnzkEV8F5ioioOEWkLG86fvbpthynjdhXYZziQ
x1hC9J2NFyi8vCTt91Fh04KGip0AaG9zuCk2wQCVyoNU3Hjezee9bq92duzzTmxsRJoy+jEZZZYo
GTKJ6SJngdJqAfRzpze0+jHreUtPc7gpBLQnIYK6BYp/uGhw9YK688eu7v95ysgshcg9qSLMo3JC
4jqLKQFBgdKDPoQ+Pltb8dUyQLpeDjeVgI6EgLIQFT5tEl3rn2losHVsexbZ3EyT9wE1uGdkIPcy
BGxn8QUq1QrA5nqW5i2tLqvrrM9NK6AdkVIvL9E9bZL/oyfMVd/jqvc8LylzRBKDJSzIExwhQzuL
QYGQj4rHfFTc8mUdu3E7yoLtbTe9gI4EqVgVkug2i5+uXGo919ixbRog+3fTbQ8qJe4ZOYNfMoTI
OoshUNosgO60AisX15aeI2PSIp5KiFLI9ubb1vV3Qb2ltwLakUCDAkWX7/nHKRmmGIl9VgYsUhJm
2NXjKYADtM1ygne9QQDIXlk49FBstMKx66D1v4+XuQr7vqTe0VcBHQlRWiOCbmmSYe2SqtL6q5rJ
zsTb7lKx3FKOYC4DoqyS/B5bvLPxvD9Qtf6saxYLQGJErmDOdOMr/zo96km1nElr8bmPOBwI9COv
HnFPRIwmkSOv9kcAS4heRsidOkpeWBgZM+UBrTFAXNYL5Vf2ii9c1trNzpYdaoVil3WIc+wdk+gQ
noie3ecCcxt9ITcLAPWt/laGEO/9U6PmzZkenTtsSMQ8uYywJVW+grCstAvCIaAdArAsIWkRDDs/
KzLm2YcjY1Lv0UdW73HabE9n6V66cxSzfEmuJssTpKGVp+0vHq73FwL46eOjpMpbRAnNmJFrGJNu
Ukf9Yrz+3rghiumCKNXXWPhLYcjxGsIpoCMsIRoFITkW8AuyM8jC1+/QLx4bozCEJIq38+1rtpR6
V/yzb8eBlRb3fo5l783N0CWolAzJHaVNzkrTzlEp2bQ2q3TC5gn6wpnoQAmwSiGh2GitnTmVMc5O
UyfKWUKCIsU7+fZDKwqdT6DDpvkzAX4/+AMFjk0tDp5GRXLpQ2MUmhgDp5gxQT8+Y7hyPsMi8uxF
71H0oebujHALECjFKaW9Lm68n18wXp2kVzIcABytD5iXFzg+WVXkegpAsOOYziqo0OkK76GyquC3
ltZAzMhhqlSNmmWTE5T6e3IN05ITFLM4GdN0vtZ3ob8Jh1NAKXFbm5PtLU/eqTSlGjkNAJjdgn/N
aedXa0tdi7+t9G0FIF49rtMSEgAs1kDLkTPO7ebm4IUWeyh1bKomXqlgMG6kJmHcSM0clYLJ8XtR
1GTnbV3F6I5wCGikAb402npp1h1s7LQUZZSMIfALFOuL3UUrfnS8+rez7v9qcold5tilgHbO1fjK
9ubb17u9oshxzMiUBKXWqJNxd+fqb0tLVs4lILFnK71H0Ind7uiPgACVcFJlrb0tV6DzxqqTIhUM
CwDf1/rrVhTa33/3pGPxJYdQ2l2cbgVcQSosdx8uqnDtbGjh9SlDVSMNWhlnilfqZk42Th2ZpLpf
xrHec5e815zrr0dfBZSwzkZfqsv+1FS1KUknUwPARVvItfKUY+cn57yP7qv07UE3p8B2uhUwLk09
e0SCOrK+hbdYHYLjRIl71wWzv9jpEoeOHhGRrJAzyEyNiJuUqX0g2sBN5kGK6y2Blp5M3lsB9Qh4
y2Ja6x6+i0ucmKgwMATwhSjdUu49tKrQ/pvN5d53ml2CGwCmJipmKjgmyuaXzNeL2a0AkQ01Th5j
2DktO3Jyk8f9vcOBQHV94OK+fPumJmvQHxJoWkaKWq9Vs+yUsbq0zGT1I4RgeH2b5wef7+c7bl8F
eKgoHVVZa8ZPEORzR6sT1BzDUAD/d9F78e2Tzv99v8D+fLVTqAKAsbGamKey1Mt9Ann4eH3gTXTz
idWtAJ8PQWOk7NzSeQn/OTHDuEikVF1R4z8BQCy+6D1aWRfY0tTGG2OM8rRoPaeIj5ZHzJxszElN
VM8K8JS5WOfv8mzRnQAKoEhmt8gyPM4lU9SmBK1MCQBnW4KONT86v1hZ1PbwSXPw4JWussVjtH9Y
NCoiL9UoH/6PSu8jFrfY2t36erQHXLIEakMi1SydmzB31h3GGXFDFNPaK8Rme9B79Ixrd0WN+1ij
NRQ/doRmuFLBkHSTOm5GruG+pFjFdAmorG4IXH1Qua6ASniclfFtDYt+oUjKipPrCQB7QBQ2lrgP
fFzm+9XWUtcqJ3/5vDLDpJ79XHZk3u8nGZ42qlj1+ydtbxysCezrydp6ugmipNJ7WBPB5tydY0jP
HaVNzs3QzeE4ZpTbI+ZbnSFPbVOw9vsfnVvqWnirPyCNGD08IlqtYkh2hjZ5dErEQzoNm+6ykyOt
Lt5/PQEuSRRKo22VkydK+vvS1XEKlhCJAnsqvcVvH7f/ZU2R67eXbMEGAMiIV5oWZWiWvz5Fv2xG
sjqNJQRvn3Rs2lji/lNP19VjAQDgD7FHhujZB9OGqYxRkZxixgRDVlqS6uEOFaJUVu0rPFzctrnF
JqijImVp8dEKVWyUXDk92zAuMZ6bFwpBU1HrOw6AdhQgUooChb0+ItMbWJitSo5Ws3IAOGEOtL53
0vHZih9sC4vtofZ7Qu6523V/fmGcds1TY3V36pUsBwAbSlxnVh2xLfAD/IAIMDf7XYIkNmXfpp2l
18rkAJAy9HKFaIr/qULkeQQKy9zf1JgDB2uaeFNGijo5QsUyacNUUTOnGO42xSnv4oOwpDi1zYkc
efUc3I5Gk6PhyTuVKaOGyLUAYPGIoY9Pu/atL/L92+4q9wbflRJ2Trpm/jPjdBtfnqB/dIThcl8A
KG7hbRuKnb8qsQsVvVlTrwQAQMUlf3kwJI24Z4JhPMtcfng5GcH49GsrxJpGvvHIaeem2ma+KSjQ
lIwUdYyCY8j4dE1KzijNnIP2llF2wcXNnsoapw9XxsgYAl6k+KzUXbi2yP3KR2ecf6z3BFsBICdW
nvnIaG3eHybqX7vbpEqUMT+9OL4Qpe8VON7dXuFd39v19FoAABRVePbGGuXTszO0P7tu6lghUonE
llRdrhArLvmKdh9u29jcFiRRkfLUxBiFNiqSU9icoZQHo5mYBI1MBgBH6wMNb+U7Pnw337H4gi1Y
ciWs+uks3Z9fztUvfzxTm9Ne8XXkvQLHNytOOZeiD4e0PgkAIAYCYknKUNUDSXEKzdWNpnil7r4p
xqkjTarZMtk/K8TQ6Qve78qqvXurGwIJqcOUKfUWHsm8KGvxSP68YudXq4pcj39X49uOK2X142O0
Tz5/u/7TVybqH0rSya6ZBwD21/gubbrgWdDgEOx9W

BokehModel(combine_events=True, render_bundle={'docs_json': {'a43f2d8f-d517-4d5c-b146-ac4a2e257406': {'version…

## Saving Simulation Runs
The `simulate` function outputs the simulation data as a `xr.Dataset`, which you can save to disk as a `*.h5` or `*.nc` file (`*.nc` is just a special kind of `*.h5` file). Alternatively, you can convert the `xr.Dataset` to a `pd.DataFrame` and save it in whatever pandas format you'd like (e.g. `*.csv`, `*.parquet`, etc). The code below shows an example, where it saves to a temporary directory.

#### **Best Practice:** `xr.Dataset` is the preferred format. If you convert to pandas, it will be a bit of a headache handling profile variables and multi-simulation datasets.

#### **Allen's Soapbox:** Saving simulation runs as data is useful in certain cases, e.g. to enable interfacing with other codes and analysis suites, but if you want to share your simulation results, it would be better to just share a notebook + git commit, as that will allow others to reproduce your results and increases traceability.

In [153]:
import os
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    # Save the dataset to NetCDF format
    dataset.to_netcdf(os.path.join(temp_dir, "data.nc"))

    # Save the dataset to HDF5 format.
    # Note that we use the same method as the netCDF format
    # This is because netCDF files are also valid HDF5 files.
    dataset.to_netcdf(os.path.join(temp_dir, "data.h5"))

    # Convert the dataset to a pandas DataFrame
    df = dataset.to_dataframe()

    # Save the DataFrame as CSV format
    df.to_csv(os.path.join(temp_dir, "data.csv"))

## Directly Providing an Interpolated Function Instead
Well ain't that nifty?

But what if manually writing out times and stuff is a bit tedious? What if you want to have some other code that generates a sequence of times and values and you want to use that instead? Sure, why not. One go-to place is the `popsim.interp` module which we use below to specify a current ramp trajectory.

In [154]:
from popsim.interp import interp

# Specify a ramp rate and create an array of plasma currents.
ramp_rate = -0.5e6
current_trajectory = 8.7e6 + ramp_rate * time_base

# Interpolate and apply the interpolated trajectory to the params struct.
new_params.plasma_current = interp(time_base, current_trajectory)

# Simulate the new trajectory.
dataset = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=new_params,
)
visualize_time_series(dataset[visualize_vars])

BokehModel(combine_events=True, render_bundle={'docs_json': {'edf903d7-4a99-445a-a5ba-219ae4b061fc': {'version…

# Running Batches of Simulations

Okay, running one simulation is good, but running a gazillion is great!

Often times, we want to run a whole bunch of simulations with different settings. There are three ways to do this:
   1) Providing a list of params.
   2) `MultiCases`: Specify multiple simulation cases.
   3) `CombinatorialCases`: Automatically generate all possible combinations of simulation cases.

As the names suggest, the former helps you specify multiple simulation cases, while the latter helps you automatically generate all possible combinations of simulation cases. Let's start providing a list of params.

Imagine you want to compare a couple of cases:
   1) Baseline
   2) Impurity injection of tungsten
   3) Impurity spike of argon
   4) ICRF coupling efficiency drops suddenly

Well, we can run these different simulatoin cases, no problem. 

In [155]:
tungsten_impurity_spike = {
    1.99: 0.0,
    2.0: 0.1,
    2.1: 0.1,
    2.11: 0.0,
}
argon_impurity_spike = {
    1.99: 0.0,
    2.0: 0.1,
    2.1: 0.1,
    2.11: 0.0,
}

icrf_drop = {
    1.99: 0.9,
    2.0: 0.5,
}


# Define baseline case as equivalent to the initial params (note the usage of replace here essentially is making a copy).
baseline_case = deepcopy(params)

# Define the tungsten case.
tungsten_case = deepcopy(baseline_case)
tungsten_case.fueling19[Impurity.Tungsten] = tungsten_impurity_spike

# Define the argon case.
argon_case = deepcopy(baseline_case)
argon_case.fueling19[Impurity.Argon] = argon_impurity_spike

# Define the ICRF drop case.
icrf_drop_case = deepcopy(baseline_case)
icrf_drop_case.fraction_of_external_power_coupled = icrf_drop

cases = [baseline_case, tungsten_case, argon_case, icrf_drop_case]


dataset = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=cases,
)
visualize_vars = [
    "aux.params.fueling19.Impurity.Tungsten",
    "aux.params.fueling19.Impurity.Argon",
    "aux.params.fraction_of_external_power_coupled",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.density_state.vol_avg_ion.Impurity.Argon",
    "state.hmode_state.hmode",
    "aux.Prad_imp_MW",
    "aux.tau_E",
]


visualize_time_series(dataset[visualize_vars])

BokehModel(combine_events=True, render_bundle={'docs_json': {'627e53ed-b56e-4ccc-8fe0-b6bdec93c583': {'version…

# Generating Random Walks and Using Multi-Case to simualte them.

Particle transport faces a good amount of uncertainty. In the current `CometMirror` model, each species particle confinement time essentially treated as `k_{species}*tau_E`. Due to the uncertainty, it seems to make sense to treat these `k` parameters as random walks.

The code below geneates 10 random walks for the `particle_confinement_time_scalar` parameters. Here, we can actually use the `visualize_config` function to see the random walks.

In [162]:
import jax

from popsim.stochastic import generate_random_walks
from popsim.visualize import visualize_config

# Diffusion mags specify the degree of randomness in the random walks.
diffusion_mags = {k: 0.25 for k in params.particle_confinement_scalar.keys()}
n_samps = 10

random_walks = generate_random_walks(
    jax.random.PRNGKey(42),  # Seed the random number generator.
    n_samps,
    time_base,
    params.particle_confinement_scalar,  # Specify a dictionary of inital conditions.
    diffusion_mags,  # Specify the degree of randomness in the random walks.
)
pprint(random_walks)
visualize_config(random_walks, time_base)

[LinearInterpolation(
  ts=f64[501],
  ys={
    <FuelSpecies.Tritium: -3>:
    f64[501],
    <FuelSpecies.Deuterium: -2>:
    f64[501],
    <Impurity.Helium: 2>:
    f64[501],
    <Impurity.Oxygen: 8>:
    f64[501],
    <Impurity.Argon: 18>:
    f64[501],
    <Impurity.Tungsten: 74>:
    f64[501]
  }
),
 LinearInterpolation(
  ts=f64[501],
  ys={
    <FuelSpecies.Tritium: -3>:
    f64[501],
    <FuelSpecies.Deuterium: -2>:
    f64[501],
    <Impurity.Helium: 2>:
    f64[501],
    <Impurity.Oxygen: 8>:
    f64[501],
    <Impurity.Argon: 18>:
    f64[501],
    <Impurity.Tungsten: 74>:
    f64[501]
  }
),
 LinearInterpolation(
  ts=f64[501],
  ys={
    <FuelSpecies.Tritium: -3>:
    f64[501],
    <FuelSpecies.Deuterium: -2>:
    f64[501],
    <Impurity.Helium: 2>:
    f64[501],
    <Impurity.Oxygen: 8>:
    f64[501],
    <Impurity.Argon: 18>:
    f64[501],
    <Impurity.Tungsten: 74>:
    f64[501]
  }
),
 LinearInterpolation(
  ts=f64[501],
  ys={
    <FuelSpecies.Tritium: -3>:
    f64[50

BokehModel(combine_events=True, render_bundle={'docs_json': {'b9e9a792-936f-4cab-896f-1441054c403c': {'version…

You may recall from earlier that `particle_confinement_scalar` is a dictionary in the 

```python
      particle_confinement_scalar={<FuelSpecies.Tritium: -3>: 3.0,
                                    <FuelSpecies.Deuterium: -2>: 3.0,
                                    <Impurity.Helium: 2>: 10.0,
                                    <Impurity.Oxygen: 8>: 10.0,
                                    <Impurity.Tungsten: 74>: 10.0},
```

But the random walk generator output a list of interpolations. But have no fear, we can call an interp function, wrap it in a `MultiCases` and it will work just fine.

Note that we can also use `MultiCases` with other variables, but we need to make sure that every `MultiCases` object has the same length or things will break. Just for the heck of it, let's also vary heating power across the random walks.

In [163]:
new_params = deepcopy(params)


# Apply the random walks to the params struct.
new_params.particle_confinement_scalar = config.MultiCases(cases=random_walks)

# Also vary heating across the cases. Note that we need to make sure all MultiCases have the same length.
heating_scales = jnp.linspace(0.8, 1.2, len(random_walks))
new_params.P_aux_MW = config.MultiCases(
    cases=[new_params.P_aux_MW * scale for scale in heating_scales]
)

dataset = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=new_params,
)
visualize_vars = [
    "aux.params.P_aux_MW",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.density_state.vol_avg_ion.FuelSpecies.Deuterium",
    "state.density_state.vol_avg_ion.FuelSpecies.Tritium",
]
visualize_time_series(dataset[visualize_vars])

/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/jax/_src/lax/lax.py:2415: RuntimeWarning: invalid value encountered in cast
  out = np.array(c).astype(eqn.params['new_dtype'])
/home/awang/repos/POPSIM/.venv/lib/python3.10/site-packages/

BokehModel(combine_events=True, render_bundle={'docs_json': {'ed534097-3298-4d16-b847-d1ca1b5d09eb': {'version…

# Running Batches of Simulations with CombinatorialCases

Okay, great! So `MultiCases` allows us to specify multiple different scenarios. But sometimes, we want to generate all possible combinations of scenarios. For example, perhaps we want to scan auxiliary heating rates for every random walk trajectory we saw above. This is where `CombinatorialCases` comes in.

When you specify `CombinatorialCases`, every list gets combined with every otehr one in the `params`. So in the example below, the number of simulations run will be the product of the number of elements in each list (3 * 2 * 4 = 24). Oh yeah, you can also specify time-dependent trajectories in both `MultiCases` and `CombinatorialCases`!
```python
params.var0 = CombinatorialCases([1, 2, 3])
params.var1 = CombinatorialCases([{0.0: 1.0, 1.0: 2.0}, {0.0: 1.0, 1.0: 3.0}]) # Specifying different time dependent trajectories
params.var2 = CombinatorialCases([500, 200, 100, 50])
```